In [1]:
import dotenv
import google.generativeai as genai
import os
import pandas as pd
import time

C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dotenv.load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

In [3]:
df = pd.read_json('product_reviews_classified.json')

In [8]:
def summary(text):
    model = genai.GenerativeModel("gemini-1.5-flash")
    resquest = "Tóm tắt feedback sau bằng tiếng việt: " + "\n" + text
    response = model.generate_content(resquest)
    return response.text


In [26]:
def transferSummaryData(start = 4050, end =5000):
    print(start)
    summary_df = pd.DataFrame(columns=['name', 'tóm tắt', 'reviews'])
    try:
        for _, product in df.iloc[start:end].iterrows():
            reviews = product['reviews']
            text = 'Tích cực: '
            for key in reviews['tích cực']:
                text += key + ', '
            text += '\n'
            text += 'Trung lập: '
            for key in reviews['trung lập']:
                text += key + ', '
            text += '\n'
            text += 'Tiêu cực: '
            for key in reviews['tiêu cực']:
                text += key + ', '

            res = summary(text)
            new_record = pd.DataFrame({'name': product['product_id'], 'tóm tắt': res, 'reviews':'reviews'}, index=[0])
            summary_df = pd.concat([summary_df, new_record], ignore_index=True)
            time.sleep(1)
    except Exception as e:
        len_df = len(summary_df)
        if len_df > 0:
            load_data(summary_df)
        else:
            len_df = 1
        time.sleep(60)
        
        transferSummaryData(start + len_df, end)

In [27]:
transferSummaryData()

2692
2693
18 dữ liệu đã được chuyển vào MongoDB thành công!
2711
15 dữ liệu đã được chuyển vào MongoDB thành công!
2726
21 dữ liệu đã được chuyển vào MongoDB thành công!
2747
20 dữ liệu đã được chuyển vào MongoDB thành công!
2767
1 dữ liệu đã được chuyển vào MongoDB thành công!
2768
96 dữ liệu đã được chuyển vào MongoDB thành công!
2864
90 dữ liệu đã được chuyển vào MongoDB thành công!
2954
52 dữ liệu đã được chuyển vào MongoDB thành công!
3006
15 dữ liệu đã được chuyển vào MongoDB thành công!
3021
21 dữ liệu đã được chuyển vào MongoDB thành công!
3042
53 dữ liệu đã được chuyển vào MongoDB thành công!
3095
177 dữ liệu đã được chuyển vào MongoDB thành công!
3272
65 dữ liệu đã được chuyển vào MongoDB thành công!
3337
87 dữ liệu đã được chuyển vào MongoDB thành công!
3424
17 dữ liệu đã được chuyển vào MongoDB thành công!
3441
137 dữ liệu đã được chuyển vào MongoDB thành công!
3578
15 dữ liệu đã được chuyển vào MongoDB thành công!
3593
48 dữ liệu đã được chuyển vào MongoDB thành công!
3641

KeyboardInterrupt: 

In [6]:
ATLAS_URI = os.getenv('MONGO_DB')
DB_NAME = 'Memories'
COLLECTION_NAME = 'summary_reviews'

In [7]:
from pymongo import MongoClient

client = MongoClient(ATLAS_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

def load_data(df):
    data_to_insert = df.to_dict(orient='records')  
    collection.insert_many(data_to_insert)  

    print(f"{len(df)} dữ liệu đã được chuyển vào MongoDB thành công!")